# 第3回：モデルの評価方法

この回は3つのパートで構成します：**モデルは本当に当たっているか ／ クラスを予測する—分類 ／ 改善実験を1つずつ行う**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼り、
説明や修正を相談します。ただし、提案されたコードは必ず実行結果を見て確かめます。

まず「基本」と「演習」を進めます。「補足」は必要に応じて読み、
「発展（任意）」「追加演習（任意）」「自由課題（任意）」は飛ばしても構いません。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回で扱うこと

学習・検証・テストを正しく分けて過学習を見抜き、回帰・分類それぞれの評価指標（Logloss・AUCの違いを含む）を使い分け、評価結果を比較・改善の判断へつなげます。

### 進め方

この回は3つのパートに分かれています。パート1から順に「基本」と「演習」を進めてください。
1日で終える必要はありません。「発展（任意）」と「追加演習（任意）」は、余裕がある場合だけ取り組みます。

### 用語について

初めて出る用語は、その用語を使うセルで説明します。ここでまとめて暗記する必要はありません。

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：モデルは本当に当たっているか

**このパートの問い：手元のスコアをどこまで信じてよいか。**


## 「手元のスコア」を疑えるようになる回

前の回で「ベースラインより高いか」を見ました。でも、その高いスコアは**信じてよいのか**？
この回のテーマはそこです。データサイエンスで最も高くつく失敗は、計算ミスではなく
**「当たっているつもり」で外すこと**。原因はたいてい次の2つです。

- **過学習**：学習データを覚えすぎ、未知データに弱い。
- **リーク（データ漏れ）**：予測時に手に入らない情報が学習に混ざり、練習だけ高得点になる。

まず、学習と検証を分けたデータを用意します（`dropna`で欠損行を落として単純化）。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

features = ["molecular_weight", "logp", "tpsa", "temperature_c", "reaction_time_h"]
clean = df.dropna(subset=features)
X_train, X_valid, y_train, y_valid = train_test_split(clean[features], clean["active"], test_size=0.25, random_state=42, stratify=clean["active"])


## 演習：木を深くすると「過学習」が見える

決定木の深さ（`max_depth`）を段階的に深くして、**学習データでの正解率**と**検証データでの正解率**を
並べます。深くするほど学習側は上がりますが、検証側はどこかで頭打ち・悪化します。この2つの差が
「覚えすぎ」の度合いです。


In [ ]:
rows = []
for depth in [1, 2, 4, 8, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    rows.append({
        "max_depth": str(depth),
        "学習スコア": accuracy_score(y_train, model.predict(X_train)),
        "検証スコア": accuracy_score(y_valid, model.predict(X_valid)),
    })
pd.DataFrame(rows).round(3)


### 出力の読み方

- `max_depth=None`（無制限）では**学習スコアが1.0近く**まで上がるのに、**検証スコアはそれほど伸びない**。典型的な過学習です。
- 検証スコアが最も高い深さの手前あたりが「ちょうど良い複雑さ」。「学習スコアの高さ」を実力だと勘違いしないことが、ここでの分かれ目です。
- 教訓：**必ず「未知データ役（検証）」で評価する**。学習データでの高得点は実力ではありません。


## 演習：リーク列を入れると「不自然に」良くなる

わざと`post_assay_signal`（活性測定後の値）を特徴量に混ぜてみます。予測したい`active`と強く連動する
ため、検証スコアが不自然に跳ね上がります。**練習では高得点なのに本番で使えない**典型です。


In [ ]:
leak_features = [*features, "post_assay_signal"]
leaked = df.dropna(subset=leak_features)
Xl_tr, Xl_va, yl_tr, yl_va = train_test_split(leaked[leak_features], leaked["active"], test_size=0.25, random_state=42, stratify=leaked["active"])
leaked_model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xl_tr, yl_tr)
print("リーク列ありの検証スコア:", round(accuracy_score(yl_va, leaked_model.predict(Xl_va)), 3))
print("post_assay_signalは測定後の値。計画時の予測には使えません。")


### 出力の読み方

リーク列を入れた検証スコアは、リークなしのときより**明らかに高い**はずです。**「スコアが急に良くなったら喜ぶ前に疑う」**。高すぎるスコアはリークの最初のサインです。第2回パート2のリーク監査と合わせて習慣にします。


## 補足：前処理も「分割の内側」で行う

見落としやすいリークが**前処理リーク**です。標準化や欠損補完を**全データで先に**行うと、検証データの
情報（平均など）が学習へこっそり混ざります。正しくは、前処理も交差検証の**各分割の内側**で学習します。
`Pipeline`に前処理を入れると、これが自動で守られます。


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

filled = clean[features].fillna(clean[features].median())
scaler_all = StandardScaler().fit(filled)               # 誤り：全データで学習
leaked_scores = cross_val_score(LogisticRegression(max_iter=1000), scaler_all.transform(filled), clean["active"], cv=5, scoring="f1")

right_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
right_scores = cross_val_score(right_pipe, clean[features], clean["active"], cv=5, scoring="f1")
print("全データ前処理(楽観的) F1平均:", round(leaked_scores.mean(), 3))
print("Pipeline内前処理(正しい) F1平均:", round(right_scores.mean(), 3))


### 出力の読み方

このデータでは差は小さいかもしれませんが、ここで確かめたいのは**やり方が正しいかどうか**そのものです。全データで前処理する方式は
原理的に楽観へ偏ります。`Pipeline`にまとめれば、分割ごとに前処理を学習し直すので安全。だから第3回パート3で
`Pipeline`を本格的に学びます。


## 自由課題（任意）：似た試料を「両側に入れない」分割

同じ化合物系列（scaffold）の似た分子が学習側と検証側の両方に入ると、検証が甘くなります（実質カンニング）。
`GroupShuffleSplit`で**系列ごとまるごと**どちらかへ振り分けると、より本番に近い評価になります。


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, valid_idx = next(splitter.split(clean, groups=clean["scaffold_group"]))
print("学習側の系列:", sorted(clean.iloc[train_idx]["scaffold_group"].unique()))
print("検証側の系列:", sorted(clean.iloc[valid_idx]["scaffold_group"].unique()))


### 出力の読み方

学習側と検証側で**系列(scaffold_group)が重ならない**ことを確認します。新規骨格への予測力を測りたいなら、
この「群を跨がせない分割」が正しい評価です。ランダム分割より点数は下がりがちですが、それが**本当の実力**です。


## 補足：学習・検証・テストの3つに分ける

ここまでは学習用と検証用の2つでした。実務では**3つ**に分けます。**検証(valid)は設定選びに何度でも使い**、
**テスト(test)は最後の1回だけ**触ります。何度も見た検証データには無意識に合わせ込んでしまうため、
「一度も見ていないテスト」で最終性能を確かめる、という役割分担です。


In [ ]:
from sklearn.model_selection import train_test_split

# まずテストを切り分け（最後まで触らない）、残りを学習用と検証用へ
work, test_set = train_test_split(clean, test_size=0.2, random_state=42, stratify=clean["active"])
train_set, valid_set = train_test_split(work, test_size=0.25, random_state=42, stratify=work["active"])
print("学習用:", len(train_set), "件（モデルを学習）")
print("検証用:", len(valid_set), "件（設定選び・改善判断に何度でも使う）")
print("テスト用:", len(test_set), "件（最後の確認まで開かない）")


### 出力の読み方

3つの件数が表示されます。**検証とテストの違い**はサイズではなく**使い方**です。検証は改善のたびに何度でも
見てよい／テストは最後に1回だけ。この分担を守ると、「検証データに合わせ込んで実力を過大評価する」失敗を
防げます（この回の振り返り「検証とテストの違いは何か」は、このセルを指させればOKです）。


## 発展（任意）：分割方式で「楽観度」はこんなに変わる

評価とは「将来の使われ方を模擬すること」。だから分割方式の選択が結果を左右します。同じモデルを
3つの分割方式（ふつうのKFold／層化／系列で分けるGroup）で評価し、スコアがどう変わるかを見ます。


In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold, GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

estimator = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, random_state=42))
X_all, y_all, groups = df[features], df["active"], df["scaffold_group"]
schemes = {
    "KFold": cross_val_score(estimator, X_all, y_all, cv=KFold(5, shuffle=True, random_state=42), scoring="f1"),
    "StratifiedKFold": cross_val_score(estimator, X_all, y_all, cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="f1"),
    "GroupKFold(系列)": cross_val_score(estimator, X_all, y_all, cv=GroupKFold(5), groups=groups, scoring="f1"),
}
pd.DataFrame({name: {"平均": s.mean(), "標準偏差": s.std(), "最低": s.min()} for name, s in schemes.items()}).T.round(3)


### 出力の読み方

- **GroupKFold（系列）の平均が最も低く**出るのが普通です。似た試料を跨がせないぶん厳しく、これが新規骨格への実力に近い。
- 「どの分割が正しいか」は**将来の使い方**で決まります。新しい系列に使うならGroup、同じ系列内での予測ならKFoldでも可。
- 標準偏差（ばらつき）も見て、平均だけで判断しません。


### ネストCV：設定選びと性能報告を分ける

`max_depth`などの設定を「検証スコアが最高になるよう」選び、その同じ検証スコアを性能として報告すると、
**出来すぎの数字**になります。これを防ぐのがネストCV：**内側のCVで設定を選び、外側のCVで評価**します。


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
param_dist = {
    "randomforestclassifier__n_estimators": [100, 200, 300],
    "randomforestclassifier__max_depth": [3, 4, 6, None],
    "randomforestclassifier__min_samples_leaf": [1, 2, 4],
}
inner = StratifiedKFold(3, shuffle=True, random_state=1)
outer = StratifiedKFold(5, shuffle=True, random_state=2)
search = RandomizedSearchCV(pipe, param_dist, n_iter=8, cv=inner, scoring="f1", random_state=42)
nested = cross_val_score(search, df[features], df["active"], cv=outer, scoring="f1")
print("ネストCVの外側F1:", nested.round(3))
print("楽観の少ない推定 平均±SD:", round(nested.mean(), 3), "±", round(nested.std(), 3))


### 出力の読み方

外側5分割それぞれで「内側で設定を選び直し→未見の外側で評価」しています。ここで出る平均が、
**設定選びの下駄を履いていない、より正直な性能**です。単純なグリッド探索の最高スコアより低めに
出るのが健全で、その差が「探索による楽観」の大きさです。


### adversarial validation：学習とテストは似ているか

もう1つの落とし穴が**分布ずれ**（学習データとテストデータの傾向が違う）です。「その行が学習か
テストか」を当てる分類器を作り、そのAUC（当てやすさ）で分布の近さを測ります。


In [ ]:
train_c = pd.read_csv(DATA / "local_competition" / "train.csv")
test_c = pd.read_csv(DATA / "local_competition" / "test.csv")
adv_features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
combined = pd.concat([
    train_c[adv_features].assign(is_test=0),
    test_c[adv_features].assign(is_test=1),
], ignore_index=True)
adv_model = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, random_state=42))
auc = cross_val_score(adv_model, combined[adv_features], combined["is_test"], cv=5, scoring="roc_auc")
print("adversarial validation AUC:", round(auc.mean(), 3))
print("0.5付近なら分布は近い。0.8以上なら分布ずれを疑う。")


### 出力の読み方

- **AUC≈0.5**：学習とテストが見分けられない＝分布が近い。手元のCVは信頼できます。
- **AUC≫0.5（0.8以上など）**：見分けがつく＝分布がずれており、手元のCVは本番を過大評価しがち。
- このデータは同じ生成過程なのでAUCは0.5付近のはず。実データでこの値が高ければ、時系列や機器差など「ずれの原因」を探します。


## 追加演習（任意）

交差検証の中身を、あえて手作りして仕組みを体で理解します。90分の外の自習向けです。
`cross_val_score`が内部でやっていることを、`for`ループで書き下します。


In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score

data = df.dropna(subset=features).reset_index(drop=True)
Xa, ya = data[features], data["active"]
k = 5
fold_id = np.arange(len(Xa)) % k          # 位置でfoldを割り当てる（デモ用）
scores = []
for f in range(k):
    is_valid = fold_id == f
    m = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xa[~is_valid], ya[~is_valid])
    scores.append(f1_score(ya[is_valid], m.predict(Xa[is_valid])))
print("手作りk-fold F1:", [round(s, 3) for s in scores])
print("平均:", round(np.mean(scores), 3))


### 出力の読み方

「4つのfoldで学習→残り1つで検証」を5回繰り返し、平均しています。これが`cross_val_score`の正体です。
中身が分かると、**分割の乱数や層化（stratify）を変えると平均が動く**ことも納得できます。


### 時系列分割：未来で過去を検証しない

`experiment_date`で並べ、`TimeSeriesSplit`で「過去で学習→未来で検証」を繰り返します。実運用が
「過去データで学習し、これから来る試料を予測する」形なら、この分割が最も現実に近い評価です。


In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

time_sorted = df.sort_values("experiment_date")
est = make_pipeline(SimpleImputer(strategy="median"), DecisionTreeClassifier(max_depth=4, random_state=42))
tscv = TimeSeriesSplit(n_splits=5)
ts_scores = cross_val_score(est, time_sorted[features], time_sorted["active"], cv=tscv, scoring="f1")
print("時系列分割F1:", ts_scores.round(3), " 平均:", round(ts_scores.mean(), 3))


### 出力の読み方

各foldは「それまでの期間」で学習し「直後の期間」で検証します。前半のfoldは学習データが少なく不安定に
なりがち。時間で性能が変わるなら、ランダム分割より厳しい（現実的な）数字が出ます。


### shuffleの有無で結果は変わる

`KFold`の`shuffle`を切り替えて比較します。データが何らかの順序（日付・バッチ順など）で並んでいると、
`shuffle=False`は偏った分割になり、スコアが不安定・楽観/悲観に振れることがあります。


In [ ]:
from sklearn.model_selection import KFold

for shuffle in [False, True]:
    kf = KFold(5, shuffle=shuffle, random_state=42 if shuffle else None)
    s = cross_val_score(est, df[features], df["active"], cv=kf, scoring="f1")
    print(f"shuffle={str(shuffle):5s}: {s.round(3)}  平均={s.mean():.3f}")


### 出力の読み方

2つの平均やばらつきが違えば、**データの並び順が結果に影響している**証拠。ふつうは`shuffle=True`が無難ですが、
時系列データでは`shuffle`してはいけません（未来が学習に混ざる）。「どう並んでいるか」を意識して分割を選びます。


---

# パート2：クラスを予測する—分類

**このパートの問い：正解率だけで十分なのはどんなときか。**


## 「正解率」だけ見ると、なぜ危ないのか

第2回パート2で「多数派と答えるだけで正解率が高くなる」ことを見ました。この回はその続きで、
**正解率（accuracy）に代わる読み方**を身につけます。鍵になるのが4つの結果です。

- **真陽性(TP)**：活性を活性と当てた／**真陰性(TN)**：非活性を非活性と当てた
- **偽陽性(FP)**：非活性を活性と誤った（無駄な追試）／**偽陰性(FN)**：活性を見逃した（機会損失）

この4つを表にしたのが**混同行列**です。まずロジスティック回帰を学習し、各試料の**活性確率**を出します。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
# グラフの日本語が文字化けしないようにする設定です。中身は今は理解しなくてOK、そのまま実行してください。
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X_train, X_valid, y_train, y_valid = train_test_split(df[features], df["active"], test_size=0.25, random_state=42, stratify=df["active"])
model = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
probability = model.predict_proba(X_valid)[:, 1]


## 演習：閾値0.5で混同行列と3指標を読む

確率が0.5以上なら「活性」と判定し、結果を混同行列で見ます。同時に3つの指標を出します。

- **precision（適合率）**：活性と判定したうち、本当に活性だった割合（＝空振りの少なさ）。
- **recall（再現率）**：本当の活性のうち、見つけられた割合（＝見逃しの少なさ）。
- **F1**：precisionとrecallのバランス（両方が高いときだけ高くなる）。


In [ ]:
prediction = (probability >= 0.5).astype(int)
print("accuracy:", round(accuracy_score(y_valid, prediction), 3))
print("precision:", round(precision_score(y_valid, prediction), 3))
print("recall:", round(recall_score(y_valid, prediction), 3))
print("F1:", round(f1_score(y_valid, prediction), 3))
ConfusionMatrixDisplay.from_predictions(y_valid, prediction, display_labels=["非活性", "活性"], cmap="Blues")
plt.title("混同行列")


### 出力の読み方

- 混同行列は**左上=TN、右下=TP**が当たり、**右上=FP、左下=FN**が外れ。色が濃い（数が多い）マスに注目します。
- **accuracyは高いのにrecallが低い**、という組み合わせが起きがちです。これは「非活性はよく当てるが、肝心の活性を見逃している」状態。活性が少ないデータでは、accuracyが良く見えてもこの罠にはまります。
- 「何を重視するか」で読む指標が変わる、というのが今回いちばん覚えておいてほしい点です。


## 演習：判定の「閾値」を動かしてみる

0.5は絶対ではありません。閾値を下げると「活性」と判定する数が増え、**recallは上がるがprecisionは下がる**、
というトレードオフが起きます。0.3・0.5・0.7で指標がどう動くか並べます。


In [ ]:
rows = []
for threshold in [0.3, 0.5, 0.7]:
    pred = (probability >= threshold).astype(int)
    rows.append({"閾値": threshold, "precision": precision_score(y_valid, pred), "recall": recall_score(y_valid, pred), "F1": f1_score(y_valid, pred)})
pd.DataFrame(rows).round(3)


### 出力の読み方

- 閾値を下げる（0.3）と**recallが上がりprecisionが下がる**、上げる（0.7）と逆。表で必ずこの向きになるはずです。
- **見逃しを避けたい場面は閾値を下げ、空振りを避けたい場面は上げる**。閾値はモデルの外側で、目的に合わせて選ぶダイヤルです（第2回パート2のコスト最適閾値につながります）。


## 補足：不均衡データではPR-AUCを見る

「閾値をいくつにするか」を決める前に、モデルの**確率の質そのもの**を1つの数字で測りたい。不均衡データ
（活性が少ない）では、ROC-AUCよりも**PR-AUC（適合率-再現率曲線の面積）**の方が実態を映します。


In [ ]:
from sklearn.metrics import average_precision_score
print("活性の割合:", round(df["active"].mean(), 3))
print("PR-AUC(平均適合率):", round(average_precision_score(y_valid, probability), 3))
print("常に多数派と予測したときのaccuracy:", round((y_valid == y_valid.mode()[0]).mean(), 3))


### 出力の読み方

- 「活性の割合」が小さいのに「多数派予測のaccuracy」が高い。**accuracyの水増し**を数字で確認できます。
- **PR-AUC**は「活性の割合」を基準線とし、それを大きく上回るほど、モデルが活性をうまく上位に並べていると読めます。閾値を決めずにモデルの良さを比べたいときの主指標です。


## 話し合い

「見逃し（FN）と空振り（FP）の、どちらがこのテーマでは高くつくか？」を5人で言葉にします。
探索段階なら見逃しを嫌ってrecall寄り、確証段階なら空振りを嫌ってprecision寄り。**正解は場面で変わります。**


## 発展（任意）：確率を「信じてよいか」と、不均衡対策

確率をコストの計算（第2回パート2）に使うなら、その確率が**較正**されている。「0.8と言ったら本当に約80%」で
ある必要があります。ここでは較正の測り方と直し方、そして少数クラスへの対処を扱います。


### 較正：予測確率と実際の頻度は一致しているか

**信頼度図**は、予測確率（横軸）に対して実際の活性率（縦軸）を描き、対角線に近いほど較正が良い、と
読みます。**Brierスコア**は較正のズレを1つの数字にしたもの（小さいほど良い）。`CalibratedClassifierCV`で
較正し直し、前後を比べます。


In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss

base_clf = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
cal_clf = CalibratedClassifierCV(base_clf, method="isotonic", cv=5).fit(X_train, y_train)
plt.figure(figsize=(6, 5))
for name, clf in {"未較正": base_clf, "較正後": cal_clf}.items():
    p = clf.predict_proba(X_valid)[:, 1]
    print(f"{name}: Brier={brier_score_loss(y_valid, p):.3f}（小さいほど良い）")
    frac, mean_pred = calibration_curve(y_valid, p, n_bins=5)
    plt.plot(mean_pred, frac, "o-", label=name)
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("予測確率"); plt.ylabel("実際の頻度"); plt.legend(); plt.title("信頼度図")
plt.tight_layout()


### 出力の読み方

- 折れ線が**対角線（点線）に近い**ほど較正が良好。対角線から膨らんでいれば、その確率帯で自信過剰／過小です。
- Brierが較正後に下がっていれば改善成功。ただし小さいデータでは較正が不安定なこともあるので、図と数字の両方で判断します。
- なお`base_clf`は「未較正」の比較用に学習しています。`CalibratedClassifierCV`は`cv=5`を指定しているため内部でモデルを学習し直します（`base_clf`の学習結果そのものは較正には使いません）。


### AUCとLoglossの違い

ここまで使ってきたPR-AUC・ROC-AUCは、確率の**順位**だけを見ています（「活性の確率が高い順に
正しく並んでいるか」）。実際の確率の値そのもの（0.51なのか0.99なのか）は問いません。

一方**Logloss（対数損失）**は、確信度まで含めて誤りを罰する指標です。正解から離れた確率を
自信満々で出すほど、罰則が大きくなります。较正されていない確率は、AUCでは分からずLoglossで
初めて悪さが見える、ということが起こります。


In [ ]:
from sklearn.metrics import roc_auc_score, log_loss

for name, clf in {"未較正": base_clf, "較正後": cal_clf}.items():
    p = clf.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, p)
    logloss = log_loss(y_valid, p)
    print(f"{name}: AUC={auc:.3f}（高いほど良い）  Logloss={logloss:.3f}（低いほど良い）")


### 出力の読み方

- **AUC**は較正の前後でほとんど変わらないはずです。较正は確率の順位を変えないため、
  順位だけを見るAUCには効果が反映されません。
- **Logloss**は較正後に下がる（改善する）ことが多いです。確信度が実態に近づいたことが、
  Loglossには反映されます。
- まとめると、**ランキング（誰から試すか）を評価したいならAUC、確率の値そのものを意思決定に
  使うならLogloss**、という使い分けになります。


### コスト行列で閾値を決める（較正済み確率で）

第2回パート2と同じ考え方を、較正した確率に適用します。見逃し(FN)が空振り(FP)の8倍高いとして、期待コストが
最小の閾値を探します。**較正済みの確率**を使うことで、コスト計算の前提が整います。


In [ ]:
import numpy as np
proba_cal = cal_clf.predict_proba(X_valid)[:, 1]
cost_fn, cost_fp = 8, 1
rows = []
for t in np.linspace(0.1, 0.9, 17):
    pred = (proba_cal >= t).astype(int)
    fp = int(((pred == 1) & (y_valid == 0)).sum())
    fn = int(((pred == 0) & (y_valid == 1)).sum())
    rows.append({"閾値": round(t, 2), "偽陽性": fp, "偽陰性": fn, "期待コスト": fp * cost_fp + fn * cost_fn})
table = pd.DataFrame(rows)
print("コスト最小の閾値:", table.loc[table["期待コスト"].idxmin(), "閾値"])
table


### 出力の読み方

見逃しのコストが高いので、最適閾値は**0.5より低め**に出ます。「0.5で判定」がいかに恣意的かを、
ここでも数字で確認できます。コスト比を変えれば最適点も動きます。


### class_weight：少数クラスの誤りを重く扱う

閾値調整とは別に、**学習の時点で**少数クラス（活性）の誤りを重く扱う方法があります。`class_weight="balanced"`は
少数クラスを自動で重み付けします。recall（見逃しの少なさ）がどう変わるかを見ます。


In [ ]:
for label, weight in {"weightなし": None, "balanced": "balanced"}.items():
    clf = make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000, class_weight=weight)).fit(X_train, y_train)
    pred = clf.predict(X_valid)
    print(f"{label:10s} F1={f1_score(y_valid, pred):.3f}  recall={recall_score(y_valid, pred):.3f}")


### 出力の読み方

`balanced`にすると**recallが上がりやすい**（活性を見つけにいく）反面、precisionやF1は下がることもあります。
「閾値で調整」と「重みで調整」は似た効果を持つ別の道具。どちらが目的に合うかを、指標を見て選びます。


## 追加演習（任意）

分類の評価を、曲線と閾値でさらに掘り下げます。90分の外の自習向けです。まず2モデルの
**適合率-再現率曲線**と**ROC曲線**を並べます。


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

candidates = {
    "ロジスティック": make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)),
    "Random Forest": make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)),
}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, est in candidates.items():
    est.fit(X_train, y_train)
    PrecisionRecallDisplay.from_estimator(est, X_valid, y_valid, ax=axes[0], name=name)
    RocCurveDisplay.from_estimator(est, X_valid, y_valid, ax=axes[1], name=name)
axes[0].set_title("適合率-再現率曲線"); axes[1].set_title("ROC曲線")
axes[1].plot([0, 1], [0, 1], "--", color="gray")
plt.tight_layout()


### 出力の読み方

- **PR曲線**は右上に張り付くほど良い。不均衡データでは、この曲線とその面積(PR-AUC)がROCより実態を映します。
- **ROC曲線**は左上に張り付くほど良い。凡例のAUCで一目比較できます。
- 2モデルの曲線が交差するなら、**どの動作点（閾値）で使うかによって優劣が変わる**ということです。


### 目標recallを満たす閾値を逆算する

「活性の見逃しは9割以上防ぎたい（recall≧0.9）」のような要件から、それを満たしつつprecisionが最大の
閾値を選びます。要件を先に決め、閾値を後から合わせる実務的なやり方です。


In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve

proba = candidates["Random Forest"].predict_proba(X_valid)[:, 1]
prec, rec, thr = precision_recall_curve(y_valid, proba)
target_recall = 0.9
ok = rec[:-1] >= target_recall
if ok.any():
    idx = np.argmax(np.where(ok, prec[:-1], -1))
    print(f"recall>={target_recall} を満たす閾値: {thr[idx]:.3f}  precision={prec[idx]:.3f}  recall={rec[idx]:.3f}")
else:
    print("目標recallを満たす点がありません")


### 出力の読み方

選ばれた閾値は0.5より低いはず（見逃しを減らすには「活性」と判定する範囲を広げる）。その代償に
precisionが下がります。**要件→閾値**の順で決めると、恣意的な0.5から卒業できます。


### 交差検証で混同行列を集計する

1回の検証ではなく、`cross_val_predict`で全データのOOF予測を作り、混同行列を集計します。1回分より
安定した内訳が見えます。


In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import confusion_matrix

oof = cross_val_predict(
    make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=1000)),
    df[features], df["active"], cv=StratifiedKFold(5, shuffle=True, random_state=42),
)
cm = confusion_matrix(df["active"], oof)
display(pd.DataFrame(cm, index=["実:非活性", "実:活性"], columns=["予:非活性", "予:活性"]))


### 出力の読み方

全420件を1件ずつ「その行を学習に使わないモデル」で予測した集計です。右上（偽陽性）と左下（偽陰性）の
大きさを比べ、**このモデルがどちらの誤りをしやすいか**を把握します。改善の方向づけに使えます。


---

# パート3：改善実験を1つずつ行う

**このパートの問い：改善した理由を後から説明できる実験とは何か。**


## 「なんとなく良くなった」を卒業する

改善は勢いでやると、後で「なぜ良くなったのか」を説明できません。この回のテーマは、**理由を後から
説明できる実験のやり方**です。次の原則が効きます。

1. **一度に変えるのは1つだけ**（複数変えると、どれが効いたか分からない）。
2. **比較条件は固定**（同じ分割・同じ指標）。
3. **結果は平均とばらつきで残す**（1回のスコアで一喜一憂しない）。
4. **良くなった実験も悪くなった実験も記録する**（消さない）。

まず、すべての実験で共通して使うデータと交差検証を用意します。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X, y = df[features], df["active"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 演習：1要素だけ変えて、実験ログに残す

`max_depth`**だけ**を変えた3つの実験を回し、結果を表（実験ログ）にします。他の設定は固定。
学習F1と検証F1平均を両方残すのは、**過学習の度合い**（第2回パート3）も一緒に記録するためです。


In [ ]:
rows = []
for depth in [3, 6, None]:
    model = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=depth, random_state=42))
    scores = cross_validate(model, X, y, cv=cv, scoring="f1", return_train_score=True)
    rows.append({"実験名": f"depth={depth}", "変更点": "max_depthのみ",
                 "学習F1": scores["train_score"].mean(), "検証F1平均": scores["test_score"].mean(),
                 "検証F1標準偏差": scores["test_score"].std()})
experiment_log = pd.DataFrame(rows)
experiment_log.round(3)


### 出力の読み方

- **検証F1平均が最も高い深さ**が候補。ただし**検証F1標準偏差**が大きいなら、その優位は不安定かもしれません。
- 差が標準偏差より小さいなら「実質同じ」と読み、より単純な（浅い）設定を選ぶのが無難です。
- `depth=None`で学習F1が跳ね上がり検証F1が伸びないなら、過学習。**表1つで「効果」と「過学習」を同時に管理**できます。


## 実験ログの最小項目

同期回でも自習でも、次を1行で残せば十分です。

- **実験名 / 変えたもの（1つ）/ 固定した比較条件 / 結果の平均とばらつき / 分かったこと / 次の仮説**

Copilotには次の実験案を出してもらってもよいですが、**優先順位と「予測時点で妥当か」の判断は人**が行います。


## 発展（任意）：探索を自動化し、正直な推定を得る

手で`max_depth`を変えるのは学習には良いですが、設定が増えると大変です。**探索の自動化**と、
第2回パート3で学んだ**ネストCV（正直な推定）**、そして**重要度を区間で読む**ことを扱います。


### RandomizedSearchCV：設定を自動で探す

複数の設定候補から無作為に組み合わせを試し、交差検証で最良を選びます。総当たり（GridSearch）より
少ない回数で広く探せるのが利点。`n_iter`が試行回数です。


In [ ]:
from sklearn.model_selection import RandomizedSearchCV

pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
param_dist = {
    "randomforestclassifier__n_estimators": [100, 200, 300],
    "randomforestclassifier__max_depth": [3, 4, 6, None],
    "randomforestclassifier__min_samples_leaf": [1, 2, 4],
    "randomforestclassifier__max_features": ["sqrt", "log2", None],
}
search = RandomizedSearchCV(pipe, param_dist, n_iter=10, cv=cv, scoring="f1", random_state=42)
search.fit(X, y)
print("最良設定:", search.best_params_)
print("探索内での最良CV F1:", round(search.best_score_, 3))


### 出力の読み方と、大事な注意

`best_params_`が選ばれた設定、`best_score_`がそのCF1です。**ただしこの`best_score_`をそのまま「性能」として
報告してはいけません**。たくさん試して一番良かった数字なので、下駄を履いています。次で正直な推定に直します。


### ネストCV：探索の下駄を脱いだ推定

「探索」を1つのモデルとみなし、その外側でもう一段の交差検証をかけます。各外側分割で設定を選び直し、
未見のデータで評価するので、**探索による楽観が乗らない正直な性能**が得られます。


In [ ]:
from sklearn.model_selection import cross_val_score

outer = StratifiedKFold(5, shuffle=True, random_state=7)
nested = cross_val_score(search, X, y, cv=outer, scoring="f1")
print("ネストCV外側F1:", nested.round(3))
print("楽観の少ない推定:", round(nested.mean(), 3), "±", round(nested.std(), 3), " ← 探索内スコアより低いのが普通")


### 出力の読み方

ネストCVの平均は、前セルの`best_score_`より**少し低い**のが普通で、その差が「探索による楽観」の大きさです。
論文や報告に載せるなら、こちらの正直な数字を使います。


### 並べ替え重要度は「区間」で読む

第1回パート1で見た並べ替え重要度を、今度は**ばらつき（±2SD）つき**で読みます。下限が0を跨ぐ特徴量は
「効いているとは言い切れない」。評価は学習に使っていない**holdout**で行い、公平性を保ちます。


In [ ]:
from sklearn.model_selection import train_test_split

X_fit, X_holdout, y_fit, y_holdout = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
best = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)).fit(X_fit, y_fit)
perm = permutation_importance(best, X_holdout, y_holdout, scoring="f1", n_repeats=30, random_state=42)
importance = pd.DataFrame({
    "特徴量": features,
    "重要度平均": perm.importances_mean,
    "下限(平均-2SD)": perm.importances_mean - 2 * perm.importances_std,
}).sort_values("重要度平均", ascending=False)
importance["0を跨ぐ"] = importance["下限(平均-2SD)"] <= 0
importance.round(4)


### 出力の読み方

- `0を跨ぐ=True`の特徴量は、**寄与があるとは断言できない**（ばらつきの範囲に0が入る）。
- 上位で`0を跨ぐ=False`の特徴量が、自信を持って「効いている」と言える列。ここから**反証可能な次の仮説**
（「この列を強める特徴量を足したら改善するのでは？」）を1つ立てて、基本の実験ログへ戻ります。これが改善実験です。


## 追加演習（任意）

実験の回し方を仕組み化します。90分の外の自習向けです。まず**実験を1行で記録する関数**を作り、
複数の設定を回してログに溜めます。手作業のコピペより、記録漏れが減ります。


In [ ]:
from sklearn.model_selection import cross_val_score

experiment_log = []
def run_experiment(name, estimator, note=""):
    "設定を交差検証で評価し、実験ログへ1行追加して返す。"
    scores = cross_val_score(estimator, X, y, cv=cv, scoring="f1")
    row = {"実験名": name, "F1平均": round(scores.mean(), 3), "F1_SD": round(scores.std(), 3), "分かったこと": note}
    experiment_log.append(row)
    return row

run_experiment("depth3", make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=3, random_state=42)), "浅め")
run_experiment("depth6", make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)), "標準")
run_experiment("leaf4", make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=6, min_samples_leaf=4, random_state=42)), "葉を大きく")
pd.DataFrame(experiment_log)


### 出力の読み方

3つの実験がログにたまり、F1平均・ばらつき・分かったことが1表に。**変更点と結果がセットで残る**ので、後から
「なぜこの設定にしたか」を説明できます。関数化しておくと、実験のたびに1行呼ぶだけで済みます。


### 検証曲線：1つの設定を動かして最適点を探す

`validation_curve`は、1つのハイパーパラメータ（ここでは`max_depth`）を動かし、学習と検証のスコア推移を
描きます。最適な複雑さが視覚的に分かります。


In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import validation_curve

depths = [2, 3, 4, 6, 8, 12]
tr, va = validation_curve(
    make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, random_state=42)),
    X, y, param_name="randomforestclassifier__max_depth", param_range=depths, cv=cv, scoring="f1",
)
plt.plot(depths, tr.mean(1), "o-", label="学習")
plt.plot(depths, va.mean(1), "o-", label="検証")
plt.xlabel("max_depth"); plt.ylabel("F1"); plt.legend(); plt.title("検証曲線")
plt.tight_layout()


### 出力の読み方

学習F1は深さとともに上がり続けますが、検証F1は途中で頭打ち・下降します。**検証F1が最大になる手前**が
最適な深さ。2本の乖離が広がるほど過学習が進んでいる、という第2回パート3の読み方がそのまま使えます。


### 実験ログをファイルに残す

ログをCSVに保存し、読み直します。セッションをまたいで実験を積み上げられ、再現性（第5回パート3）にもつながります。


In [ ]:
out = ROOT / "workspace" / "experiment_log.csv"
pd.DataFrame(experiment_log).to_csv(out, index=False)
reloaded = pd.read_csv(out)
print("保存＆再読込した実験ログ:", out)
display(reloaded)


### 出力の読み方

`workspace/experiment_log.csv`に保存され、読み直しても同じ内容。**記録を残す文化**が、思いつきの改善を
再現可能な知見へ変えます。良い変更も悪い変更も、まずログに残すことを、この回でいちばんの習慣にしてください。


---

## よくある誤り

- 前処理を全データで済ませてから分割する
- 同じ系列の類似化合物を両側へ入れる
- 検証データを何度も見て実質的に学習する
- 常に閾値0.5を使う
- 偽陽性と偽陰性のコストを同じとみなす
- 未較正の確率をそのまま意思決定へ使う
- 同時に複数要素を変える
- 探索に使った分割で最終性能も報告する
- 悪化した実験を記録から消す

## 自習（任意・30〜60分）

- KFold・StratifiedKFold・GroupKFoldのF1分布を箱ひげ図で比べる
- adversarial validationのAUCを下げる列を1つ見つけ理由を書く
- CalibratedClassifierCVで較正前後の信頼度図とBrierスコアを比べる
- コスト行列から期待コスト最小の閾値を求め、0.5と比較する
- RandomizedSearchCVの最良設定を、ネストCVの外側スコアで確かめる
- 並べ替え重要度を20反復で計算し、区間が0を跨ぐ列を挙げる

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 検証とテストの違いは何か
2. ネストCVが必要になるのはどんなときか
3. adversarial validationのAUCが高いと何を意味するか
4. accuracyが危険な例は何か
5. 較正が悪い確率を使うと何が起きるか
6. コストから閾値をどう決めるか
7. 1要素だけ変える理由は何か
8. ネストCVは何を防ぐか
9. 重要度の区間が0を跨ぐとどう解釈するか

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
